In [9]:
from datetime import date
from pathlib import Path

import blpapi
import pandas as pd

TENORS = {
    "2Y": "USGG2YR Index",
    "5Y": "USGG5YR Index",
    "10Y": "USGG10YR Index",
    "30Y": "USGG30YR Index",
}
SECURITY = list(TENORS.values())
FIELD = ["PX_LAST"]
START_DATE = date(2026, 1, 1)
END_DATE = date(2026, 8, 19)
CURVE_DATES = [date(2026, 1, 2), date(2026, 4, 1), date(2026, 8, 19)]
DATA_DIR = Path("data")


def _element_value(row, field):
    if not row.hasElement(field) or row.getElement(field).isNull():
        return None
    return row.getElementAsFloat(field)


def fetch_historical_prices(
    securities=SECURITY,
    fields=FIELD,
    start_date=START_DATE,
    end_date=END_DATE,
):
    """Pull daily Bloomberg history for every security and field provided."""
    if not securities:
        raise ValueError("securities must contain at least one Bloomberg security")
    if not fields:
        raise ValueError("fields must contain at least one Bloomberg field")
    if not isinstance(start_date, date) or not isinstance(end_date, date):
        raise TypeError("start_date and end_date must be datetime.date objects")
    if start_date > end_date:
        raise ValueError("start_date must be on or before end_date")

    session = blpapi.Session()
    if not session.start():
        raise RuntimeError("Could not start Bloomberg session. Is Bloomberg Terminal running?")

    try:
        if not session.openService("//blp/refdata"):
            raise RuntimeError("Could not open Bloomberg reference-data service.")

        request = session.getService("//blp/refdata").createRequest("HistoricalDataRequest")
        for security in securities:
            request.getElement("securities").appendValue(security)
        for field in fields:
            request.getElement("fields").appendValue(field)
        request.set("startDate", start_date.strftime("%Y%m%d"))
        request.set("endDate", end_date.strftime("%Y%m%d"))
        request.set("periodicitySelection", "DAILY")
        session.sendRequest(request)

        records = []
        while True:
            event = session.nextEvent()
            for message in event:
                if not message.hasElement("securityData"):
                    continue
                security_data = message.getElement("securityData")
                security = security_data.getElementAsString("security")
                field_data = security_data.getElement("fieldData")
                for index in range(field_data.numValues()):
                    row = field_data.getValueAsElement(index)
                    record = {
                        "security": security,
                        "date": row.getElementAsDatetime("date"),
                    }
                    record.update({field.lower(): _element_value(row, field) for field in fields})
                    records.append(record)
            if event.eventType() == blpapi.Event.RESPONSE:
                break

        if not records:
            return pd.DataFrame(columns=["security", "date", *[field.lower() for field in fields]])
        return pd.DataFrame(records).set_index(["security", "date"]).sort_index()
    finally:
        session.stop()


prices = fetch_historical_prices(
    securities=SECURITY,
    fields=FIELD,
    start_date=START_DATE,
    end_date=END_DATE,
)
DATA_DIR.mkdir(parents=True, exist_ok=True)
data_path = DATA_DIR / "yield_curve_data.csv"
prices.to_csv(data_path)
print(f"Saved {len(prices):,} rows to {data_path}")


Saved 656 rows to data\yield_curve_data.csv


In [14]:
from IPython.display import HTML, display
import plotly.express as px

plot_data = prices.reset_index()
plot_data["tenor"] = plot_data["security"].map({security: tenor for tenor, security in TENORS.items()})
plot_data["date"] = pd.to_datetime(plot_data["date"])
plot_data["tenor_order"] = plot_data["tenor"].map({tenor: index for index, tenor in enumerate(TENORS)})
plot_data = plot_data.sort_values(["date", "tenor_order"])

history_figure = px.line(
    plot_data,
    x="date",
    y="px_last",
    color="tenor",
    markers=True,
    category_orders={"tenor": list(TENORS)},
    title="US Treasury Yields Through Time",
    labels={"px_last": "Yield / PX_LAST", "date": "Date", "tenor": "Tenor"},
)
display(HTML(history_figure.to_html(include_plotlyjs="cdn", full_html=False)))

available_dates = plot_data["date"].dt.normalize().drop_duplicates().sort_values().tolist()
snapshot_rows = []
for target_date in CURVE_DATES:
    nearest_date = min(available_dates, key=lambda value: abs(value - pd.Timestamp(target_date)))
    snapshot_rows.append(
        plot_data[plot_data["date"].dt.normalize() == nearest_date].assign(
            snapshot_date=nearest_date.date()
        )
    )

snapshots = pd.concat(snapshot_rows, ignore_index=True)
curve_figure = px.line(
    snapshots,
    x="tenor",
    y="px_last",
    color="snapshot_date",
    markers=True,
    category_orders={"tenor": list(TENORS)},
    title="US Treasury Yield Curve Shape",
    labels={"px_last": "Yield / PX_LAST", "tenor": "Tenor", "snapshot_date": "Snapshot date"},
)
display(HTML(curve_figure.to_html(include_plotlyjs=False, full_html=False)))
